# SARBLOH alignment on TPU: LoRA SFT of the 27B with Tunix

Trains a LoRA adapter for `Qwen/Qwen3.8-27B` on our authored traces (`Alignment/data/*.jsonl`) on a Kaggle TPU v5e-8,
so the RTX Pro 6000 stays free. The code is `Alignment/lora_sft.py` with `backend: "tunix"` (`Alignment/lora_sft_tunix.py`):
the same rendering and loss masks as the GPU run, then Tunix's trainer on a MaxText model.

Steps: install MaxText + Tunix → import `Alignment/` from the repo dataset → download the HF weights → convert them to
MaxText once (CPU, ~2 bytes/param on disk outside `/kaggle/working`) → eval loss before → train → eval loss after →
export a PEFT adapter (`sft_out/adapter/`) that vLLM/SGLang load on the GPU for the game eval.

Attach: `banwait13/sarblohagent` (this repo; update it after committing, it must contain `Alignment/lora_sft_tunix.py`
and `Alignment/data/*.jsonl`). Accelerator **TPU v5e-8**, internet **on**. Push with
`uv run python scripts/kaggle_push.py alignment-tpu`; then `alignment-tpu-status` / `alignment-tpu-output`.

To resume after a time-limited run: attach the previous version's output and set `"resume_from"` in `CONFIG` to its
`sft_out/checkpoint` folder.

In [ ]:
## CONFIG: every knob of the run (lora_sft.DEFAULTS; the TPU ones are lora_sft_tunix.TUNIX_DEFAULTS)
import time
NOTEBOOK_START = time.time()

AGENT_DATASET = "banwait13/sarblohagent"   # the repo, mirrored from git
MAXTEXT_REF = "42a331c9dad2086004d5de2a71b99e2db8caa531"   # MaxText main of 2026-10-13: dense qwen3.5 (not in 0.2.5)
MODEL = "Qwen/Qwen3.8-27B"                 # bf16 HF weights, downloaded (~54 GB); "google/gemma-4-31b-it" also maps

CONFIG = {
    "backend": "tunix",
    "model": MODEL,
    "data_path": None,                     # set below to <repo dataset>/Alignment/data/*.jsonl
    "select": "all",
    "filter_cleared": False,
    "max_seq_len": 8192,                   # one example per assistant turn, its context cut from the left to fit
    "mask_mode": "per_turn",
    "overlong": "left",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "lr": 1e-4,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "save_every": 25,                      # steps between checkpoints (LoRA + optimizer, small)
    "max_train_hours": 6.0,                # TPU sessions end at 9 h (UNCONFIRMED); download + convert + export first
    "eval_max_examples": 64,
    "log_every": 1,
    "gen_samples": 0,                      # no generation on the TPU; the game eval runs on the GPU
    "output_dir": "/kaggle/working/sft_out",
    "tunix": {
        "fsdp": -1,                        # all 8 chips; global batch = per_device_batch x 8
        "tensor": 1,
        "per_device_batch": 1,
        "vocab_tiling": 4,
        "remat_policy": "full",
        "lora_module_path": "auto",        # qwen3_5: MLP only (gate/up/down_proj in the exported adapter)
        "work_dir": "auto",                # weights + converted copy, outside /kaggle/working
        "resume_from": None,               # e.g. "/kaggle/input/<previous output>/sft_out/checkpoint"
        "export_hf": True,
    },
}
print(CONFIG)

In [ ]:
## 1. Environment, then MaxText + Tunix (post-training extra: tunix, qwix, orbax; CPU torch for the conversion)
import subprocess, sys
for cmd in (["python", "--version"], ["df", "-h", "/", "/tmp", "/kaggle/working"], ["free", "-g"], ["nproc"]):
    print(subprocess.run(cmd, capture_output=True, text=True).stdout)
assert sys.version_info >= (3, 12), f"MaxText needs Python >= 3.12, this image has {sys.version}"

t = time.time()
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                       "--extra-index-url", "https://download.pytorch.org/whl/cpu",
                       f"maxtext[tpu-post-train] @ git+https://github.com/AI-Hypercomputer/maxtext@{MAXTEXT_REF}"])
print(f"installed in {time.time() - t:.0f}s")
import importlib.metadata as md

def _version(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

print({p: _version(p) for p in ("maxtext", "google-tunix", "qwix", "jax", "jaxlib", "libtpu", "flax", "transformers")})
import jax
print(jax.default_backend(), jax.device_count(), jax.devices())
assert jax.default_backend() == "tpu", "no TPU: set the accelerator to TPU v5e-8"

In [ ]:
## 2. The pipeline and the data from the repo dataset (nothing is embedded in this notebook)
from pathlib import Path

owner, slug = AGENT_DATASET.split("/")
candidates = [Path("/kaggle/input/datasets") / owner / slug, Path("/kaggle/input") / slug]
REPO = next((p for p in candidates if (p / "Alignment" / "lora_sft_tunix.py").exists()), None)
if REPO is None:
    hits = sorted(Path("/kaggle/input").glob("**/Alignment/lora_sft_tunix.py"))
    REPO = hits[0].parent.parent if hits else None
if REPO is None:
    raise FileNotFoundError(f"Alignment/lora_sft_tunix.py not found under /kaggle/input: attach {AGENT_DATASET} "
                            "and update it from git after committing the file")
ALIGN = REPO / "Alignment"
DATA = sorted((ALIGN / "data").glob("*.jsonl"))
print(REPO, [p.name for p in DATA])
if not DATA:
    raise FileNotFoundError(f"no {ALIGN / 'data'}/*.jsonl in the dataset: commit Alignment/data/*.jsonl and update it")
sys.path.insert(0, str(ALIGN))
CONFIG["data_path"] = str(ALIGN / "data" / "*.jsonl")
import lora_sft, lora_sft_tunix
print(lora_sft.__file__, lora_sft_tunix.__file__)

In [ ]:
## 3. Run: download -> convert -> eval before -> train -> eval after -> export the adapter
# Always writes sft_out/result_card.{json,md} and train.log, also on a failure (the card names the failing step).
CARD = lora_sft.run(CONFIG)
print(f"VERDICT {CARD['verdict']} after {(time.time() - NOTEBOOK_START) / 3600:.2f} h of notebook time")

In [ ]:
## 4. Result card and outputs (the adapter in sft_out/adapter/ is what the GPU eval loads)
from IPython.display import Markdown, display
out = Path(CONFIG["output_dir"])
display(Markdown((out / "result_card.md").read_text(encoding="utf-8")))
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:10.1f} MB  {p.relative_to(out)}")